# Polarimetric orientation compensation

- Open the San Francisco ALOS-1 scattering matrix.
- Convert `S` to `T3`.
- Apply 4 × 1 multilooking to reduce the number of azimuth lines.
- Estimate and compensate the polarimetric orientation.
- Display the orientation angle and corrected Pauli RGB image.

In [ ]:
from pathlib import Path

import dask
import matplotlib.pyplot as plt
from dask.diagnostics import ProgressBar

from polsarpro.io import open_netcdf_beam
from polsarpro.polarisation import orientation_compensation
from polsarpro.util import S_to_T3, multilook, pauli_rgb

input_file = Path("/data/psp/test_files/SAN_FRANCISCO_ALOS1_slc.nc")

In [ ]:
S = open_netcdf_beam(input_file)
S

## Prepare the coherency matrix

Four azimuth looks reduce the number of image lines before orientation estimation and correction.

In [ ]:
T3 = S_to_T3(S)
T3 = multilook(T3, dim_az=4, dim_rg=1)
T3

## Estimate and compensate the orientation

The function returns the corrected T3 matrix and the angle estimated with a 3 × 3 spatial window.

In [ ]:
corrected_lazy, orientation_lazy = orientation_compensation(
    T3, boxcar_size=(3, 3)
)
pauli_lazy = pauli_rgb(corrected_lazy)

with ProgressBar():
    corrected, orientation, pauli = dask.compute(
        corrected_lazy, orientation_lazy, pauli_lazy
    )

## Display the result

In [ ]:
figure, axes = plt.subplots(1, 2, figsize=(11, 7), constrained_layout=True)
angle = axes[0].imshow(
    orientation.orientation_angle,
    cmap="twilight",
    vmin=-22.5,
    vmax=22.5,
    interpolation="none",
)
axes[0].set_title("Orientation angle")
figure.colorbar(angle, ax=axes[0], label="Degrees", shrink=0.75)
axes[1].imshow(
    pauli.transpose("y", "x", "band"), interpolation="none"
)
axes[1].set_title("Corrected Pauli RGB")
for axis in axes:
    axis.axis("off")
plt.show()